# Adatptive AI Tutor

## Build an adaptive AI Tutor with Gradio interface for Multi-level learning.
- Build an AI Tutor to deliver multi-level explanations. Akey feature will be a complexity slider, allowing user to adjust the explanation level - from "Explain like i'm 5 years old to expert or PhD level
- Design a user friendly web interface using Gradio, a powerful library for building interactive applications.
- Finally, implement streaming responses to enhance ther user experience by displaying answer in real time.

## Learning Objectives.
- Build an application - AI Tutor - That can adjust its input parameters to control the complexity of the output - Tutor Outcome (from explaining to a 5-year old to an expert)
- Master "Gradio" to create powerful web interfaces for the applications.
- Implement Streaming responses in the Gradio app for a better user experience.

### Gradio
- Gradio is the fastest way to demo the applications - Machine learning, llm models - with a friendly web interface so that anyone can use it, anywhere.
- It comes with following benefits.
    - **Easy Setup and Installation**.
        Install with pip and create interfaces in just a few lines - works with any Python function or library.
    - **Instant Sharing**.
        Embed in notebooks or launch as a web app with a shareable public link for remote access.
    - **Permanent Hosting**.
        Deploy on Hugging Face Spaces for long-term access and a dedicated URL.

In [1]:
# In previous notebook the interaction with LLMs - OpenAI's GPT - was with using their API.
# Now, move a step further
# Share AI creations with other through a simple web interface
# Use Gradio, Gradio is an amazing python library that allow you to build user interfaces (UIs) quickly.
# for machine learning models, APIs, or any python function.

# Install necessary libraries if not already installed.
!pip install -q --upgrdae openai python-dotenv gradio

Could not import runpy._run_module_as_main
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1342, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 938, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1179, in exec_module
  File "<frozen runpy>", line 14, in <module>
  File "C:\Users\Paco_Minha\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\importlib\__init__.py", line 57, in <module>
    import warnings
  File "C:\Users\Paco_Minha\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 591, in <module>
    filterwarnings("default", category=DeprecationWarning,
  File "C:\Users\Paco_Minha\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 155, in filterwarnings
    import re
  File "C:\Users\Paco_Minha\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\re\__init__.py"

In [2]:
# Import necessary libraries
import os
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
openai_api_key = os.getenv("OPENAI_API_KEY")

print("OpenAI API keyloaded successfully!")
print(f"Key start with : {openai_api_key[:10]}")

# Configure the openai's client using the loaded key
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client configured.")

OpenAI API keyloaded successfully!
Key start with : sk-proj-3Z
OpenAI client configured.


In [3]:
# Define a helper function to display markdown nicely.
def prnit_markdown(text):
    display(Markdown(text))

### Build a Basic AI Tutor Function (No Gradio yet)
Before building the uer interface, create the core python functino that will act as out AI Tutor.  
This function will:
1. Take a user's `question` as an input.
2. Construct a prompt for the OpenAI API, telling it to act as a helpful tutor.
3. Call the OpenAI API.
4. Return the AI's Answer.

In [18]:
# Define the python function to get a response from the AI Tutor
def get_ai_tutor_response(user_question):
    """
    Sends a question to the OpenAI API, asking it to respond as an AI Tutor.
    
    Args:
        user_question (str): The question asked by the user.
    Returns:
        str: The AI's response, or an error message.
    """

    # Define the system prompt - instruction for the AI's personality and role.
    system_prompt = "You are a Patient AI Tutor. Explain concepts clearly and concisely."

    try:
        # make the API call to OpenAI
        response = openai_client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = [{
                'role' : 'system', 'content' : system_prompt},
                {'role' : 'user', 'content' : user_question,
            
            }],
            temperature = 0.7, # allows for some creativity but keeps response focused.

        )
        ai_response = response.choices[0].message.content
        return ai_response
    except Exception as e:
        print(f"An error occured: {e}")

In [19]:
# Let's test function with a sample question.
test_question = "Could you explain the concept of functions in Python and their purpose in programming?"
prnit_markdown(f"Asking the AI Tutor: \n {test_question}")

# call the function and get the response
tutor_answer = get_ai_tutor_response(test_question)

print(f"\n🤖 AI Tutor's Response:\n\n {tutor_answer}")

Asking the AI Tutor: 
 Could you explain the concept of functions in Python and their purpose in programming?


🤖 AI Tutor's Response:

 Certainly! In Python, a function is a block of reusable code that performs a specific task. Functions help organize and modularize code, making it easier to read, maintain, and debug. Here are the key concepts related to functions in Python:

### 1. Definition
A function is defined using the `def` keyword, followed by the function name and parentheses. Inside the parentheses, you can specify parameters (inputs) that the function can accept. The body of the function contains the code that runs when the function is called.

**Example:**
```python
def greet(name):
    print(f"Hello, {name}!")
```

### 2. Calling a Function
To execute a function, you "call" it by using its name followed by parentheses, optionally passing in any required arguments.

**Example:**
```python
greet("Alice")  # Output: Hello, Alice!
```

### 3. Parameters and Arguments
- **Parameters** are the variables listed in the function's definition. They act as placeholders for the values you pas

### Build an Interactive Interface using Gradio

Use Gradio to wrap the get_ai_tutor_response function in a simple web interface.  
**Core Gradio Concepts**:   
`gr.Interface`  
The `gr.Interface` class is the main way to build UIs in Gradio. It requires
- `fn`: The Python function to call (`get_ai_tutor_response`).
- `inputs`: What kind of input component(s) the user will use (e.g., a text box, a slider)
- `outputs`: What kind of output component(s) will be used to display the results (e.g., another text box)
- `title`, `description`: Optional text to display on the UI.  
Finally, call the `.launch()` method on interface object to start the web server and display the UI.

In [21]:
# Install and Import Gradio
# !pip install gradio
import gradio as gr

c:\AI_DataScience_Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
# Define the Gradio interface
# fn: The function to wrap (out AI tutor function)
# inputs: A component for the user to type their question
# outputs: A component to display the AI's response
# title/Description: Text for the UI heading

ai_tutor_interace_simple = gr.Interface(
    fn = get_ai_tutor_response,
    inputs = gr.Textbox(lines = 3,
                        placeholder="Ask the AI tutor anything.....",
                        label= "Your Question"),
    outputs = gr.Textbox(label = "AI Tutor's Answer"),
    title = "Your Personal AI Tutor 🤖 ",
    description= "Enter your question below and the AI Tutor will provide an explanation. Powerd by OpenAI",
    # allow_flagging = "never"

)

# Launch the interface!
# This will typically create a link (or display inline in environments like Google Colab/Jupyter)

print("Launching Gradio interface....")
ai_tutor_interace_simple.launch()



Launching Gradio interface....
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [29]:
### Add Streaming for Enhanced chat Experience

It takes a while before you see anything until the AI finishes generating the entire response. For longer answers, this can feel slow.  
Use "streaming" to improve the user experience. This helps process the response chunk-by-chunk as it arrives.  
Gradio natively supports Python Generator function for streaming output to text boxes!.  
**Steps**
1. Modify AI interaction fuction to use `stream = True` in the API call.
2. Make the function a gnerator by `yield` to return each chunk of text as it comes in, instread of returning the whole response at the end.
3. Update the `gr.Interface` to use this new streaming function. Gradio hanles the rest

In [32]:
# Create a new function that streams the response

def stream_ai_tutor_response(user_question):
    """
    Sends a question to the OpenAI API and streams the response as a generator.
    Args:
        user_question (str) : The question asked by the user.
    Yields:
        str: Chunks of the AI's response.
    """

    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly an dconcisely."

    try:
        # only change here is "Stream = True"
        stream = openai_client.chat.completions.create(
            model = 'gpt-4o-mini',
            messages = [{"role": "system", 'content' : system_prompt},
                {"role" : 'user', 'content' : user_question}],
            temperature= 0.7,
            stream = True,
        )

        # Iterate through the response chunks
        full_response = ""

        for chunk in stream:
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                # Extract the text from this chunk
                text_chunk = chunk.choices[0].delta.content
                # Add this chunk to our growing response
                full_response += text_chunk

                yield full_response
    except Exception as e:
        print(f"An error occured during streaming: {e}")
        yield f" Sorry, I encountered an error : {e}"


In [34]:
# Create a Gradion interface using the Streaming Function.
# Notice the fn points to the new 'stream_ai_totor_response' function.  The rest is the same!
ai_tutor_interface_streaming = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = gr.Textbox(lines = 3, 
                        placeholder= "Ask The AI Tutor anything...",
                        label = "Your Question"),
    outputs= gr.Markdown(
        label = "AI Tutor's Answer (Streaming)",
        container = True,
        height = 250,
    ),  # Output is still a Markdown (it renders as HTML), container lets it be scrollable is set to 250px (for better visibility)
    title = "🤖 AI Tutor with Streaming",
    description = "Enter your Question. The answer will appear word-by-word!",
    # allow_flagging = 'never',
)

# Launch the streaming interface
print("Launching Streaming Gradio Interface.....")
ai_tutor_interface_streaming.launch()

Launching Streaming Gradio Interface.....
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


### Adding An Explanation Level Slider

Our AI Tutor is helpful, but what if the user needs a simpler explanation, or perhaps a more in-depth one?.  
We can add a control for this!  
Gradio provides various input components. Let's use a `gr.Slider` to let the user choose an explanation level.  
**Steps**:
1. Define a mapping from slider values (e.g., 1 to 5) to descriptive levels ("like I'm 5", "like I'm 10", "high school", "college", "expert").
2. Modify streaming function to accept this `level` as an additional input.
3. Inside the function, use the selected level to modify the system promp sent to OpenAI, instructing the AI on the desired explanation complexity.
4. Update the `gr.Interface` to include the `gr.Slider` in the `inputs` list. Remember `inputs` can be a list of multiple components!  
**Test The Slider!**
1. Ask the same question (e.g., "What is elctricity")
2. First, try it with the slider set to `1` ("Like I'm 5 years old"). Observe the response.
3. Then, ask the same question again, but move the slider to `5` ("like an expert")
4. Compare the two responses. You should see a significant difference in vocabulary, depth, and complexity! The DEBUG print statement above the API call in the code will also show you the exact system prompt being used.

In [36]:
# Define the mapping for explanation levels
explanation_levels = {
    1: "Like I'm 5 years old",
    2: "Like I'm 10 years old",
    3: "Like a high school student",
    4: "Like a college student",
    5: "Like an expert in the field",
}


In [40]:
# Create a new function that accepts question and level and streams the response.
def stream_ai_tutor_response_with_level(user_question, explanation_level_value):
    """
    Streams AI Tutor response based on user questino and selected explanation level.
    
    Args:
        user_question (str): The question from the user.
        explanation_level_value (int): The question from the slider (1-5).
        
    Yields: 
        str: chunks of the AI's response.
    """

    # Get the descriptive text for the chosen level
    level_description = explanation_levels.get(
        explanation_level_value, "clearly and concisely"

    ) # Default if level not found

    # Construct the system prompt dynamically based on the level.

    system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level_description}"
    print(f"DEBUG: Using System Prompt: '{system_prompt}'")

    try:
        stream = openai_client.chat.completions.create(
            model = 'gpt-4o-mini',
            messages= [{'role' : 'system', 'content' : system_prompt},
                       {'role' : 'user', 'content' : user_question},
            ],
            temperature=         0.7,
            stream = True,
        )

        # Iterate through the response chunks
        full_response = "" # keep track of the full response if needed.

        for chunk in stream:
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                text_chunk = chunk.choices[0].delta.content
                full_response += text_chunk

                yield full_response
    except Exception as e:
        print(f"An eeror occured during streaming : {e}")
        


In [ ]:
ai_tutor_interface_slider = gr.Interface(fn = stream_ai_tutor_response_with_level,
                                         inputs = [gr.Textbox(lines=3, placeholder="Ask the AI Tutor a question ....", label = "Your Question"),
                                                   gr.Slider(
                                                    minimum=1,
                                                    maximum=5,
                                                    step =1,    # Allows whole numbers
                                                    value= 3,   # Default level (High school)
                                                    label = "explanation_level",
    ),
    ],
    outputs = gr.Markdown(label = "AI TUtor's Explanation (Streaming.)", 
                          container= True, 
                          height = 250 ),
    # allow_flagging = "never",
    description= "Ask a question and select the desired level of explanation using the slider.",
)

print(f"Launching Advanced Gradio Interface with Slider....")
ai_tutor_interface_slider.launch()

Launching Advanced Gradio Interface with Slider....
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DEBUG: Using System Prompt: 'You are a helpful AI Tutor. Explain the following concept Like a college student'
DEBUG: Using System Prompt: 'You are a helpful AI Tutor. Explain the following concept Like I'm 5 years old'
